In [68]:
import os

print("Current working directory:", os.getcwd())

Current working directory: c:\Users\Babak.Baradaranhezav\ml-projects\ml_airbnb_price_regression


In [69]:
from pathlib import Path
import os

# Set working directory only if not already set
cwd = Path.cwd()
if not (cwd / ".git").exists():
    for parent in cwd.parents:
        if (parent / ".git").exists():
            os.chdir(parent)
            print("Working directory set to repo root:", parent)
            break
    else:
        raise FileNotFoundError("Could not find .git repo root. Are you inside the correct project folder?")
else:
    print("Already in repo root:", cwd)


Already in repo root: c:\Users\Babak.Baradaranhezav\ml-projects\ml_airbnb_price_regression


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define feature lists based on predict.py and engineered features 
# Define initial feature lists
numerical_features = [
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'minimum_nights', 'maximum_nights',
    'number_of_reviews', 'review_scores_rating'
]

categorical_features = [
    'host_is_superhost', 'instant_bookable', 'room_type', 'neighbourhood_cleansed'
]

binary_amenities = []

# Note: In feature_engineering, host_flags should reflect binary columns created
host_flags = {
    'host_has_profile_pic': 'host_has_profile_pic_binary',
    'host_identity_verified': 'host_identity_verified_binary'
}

In [71]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load cleaned EDA data
try:
    df = pd.read_csv('data/processed/cleaned_listings.csv')
    print(f"Loaded dataset with shape: {df.shape} which means {df.shape[0]} rows and {df.shape[1]} columns.")
    if df.shape[1] > 100:  # Arbitrary upper limit
        print("Warning: Unexpectedly large number of columns.")
except FileNotFoundError:
    print("Error: cleaned_listings.csv not found.")
    raise

Loaded dataset with shape: (5090, 80) which means 5090 rows and 80 columns.


## Step 1: Feature Engineering – Host Age

**Problem**  
`host_since` is a date column but stored as text. We want to understand how experienced a host is.

**Goal**  
Calculate how long a host has been active (in days).

**Approach**  
Convert `host_since` to datetime, and subtract from today to get `host_age_days`.

In [ ]:
# Drop rows with missing price
df = df.dropna(subset=["price"])

# Add interaction terms (accommodates_bedrooms and min_nights_reviews)
df['accommodates_bedrooms'] = df['accommodates'] * df['bedrooms']
df['min_nights_reviews'] = df['minimum_nights'] * df['number_of_reviews']
numerical_features.extend(['accommodates_bedrooms', 'min_nights_reviews'])

# Add season feature (if last_review is available)
if 'last_review' in df.columns:
    df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')
    df['season'] = df['last_review'].dt.month % 12 // 3  # 0=Winter, 1=Spring, 2=Summer, 3=Fall
    numerical_features.append('season')

In [ ]:
# Add host flag binaries
for orig_col, new_col in host_flags.items():
    if orig_col in df.columns:
        df[new_col] = df[orig_col].map({'t': 1, 'f': 0}).fillna(0).astype(int)
        binary_amenities.append(new_col)

In [ ]:
# Add amenities as binary features (assuming amenities_list from a previous step)
if 'amenities' in df.columns:
    amenities_list = ['Wifi', 'Kitchen', 'Heating', 'TV', 'Essentials', 'Hair Dryer', 'Iron', 'Free Parking', 'Hangers', 'Laptop Friendly Workspace']
    for amenity in amenities_list:
        df[f'Has_{amenity.replace(" ", "_")}'] = df['amenities'].str.contains(amenity, case=False, na=False).astype(int)
        binary_amenities.append(f'Has_{amenity.replace(" ", "_")}')

In [ ]:
# Location-based features
# One-hot encode neighbourhood_cleansed as sub-region dummies
df = pd.get_dummies(df, columns=['neighbourhood_cleansed'], drop_first=True)

# Add geospatial feature: distance to city center (if latitude/longitude available)
if 'latitude' in df.columns and 'longitude' in df.columns:
    from math import radians, sin, cos, sqrt, atan2
    def haversine_distance(lat1, lon1, lat2, lon2):
        R = 6371  # Earth radius in kilometers
        lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
        c = 2 * atan2(sqrt(a), sqrt(1-a))
        distance_km = R * c
        return distance_km

    city_center_lat, city_center_lon = 49.2827, -123.1207  # Vancouver city center (adjust if different)
    df['distance_to_center_km'] = df.apply(
        lambda row: haversine_distance(row['latitude'], row['longitude'], city_center_lat, city_center_lon),
        axis=1
    )
    numerical_features.append('distance_to_center_km')

In [ ]:
# Update all_features to include new numerical features, binary amenities, and neighbourhood dummies
all_features = numerical_features + categorical_features + binary_amenities
all_features.extend([col for col in df.columns if col.startswith('neighbourhood_cleansed_')])

In [ ]:
# Drop unneeded columns
drop_cols = ["id", "name", "host_id", "host_name", "last_review", "license", "latitude", "longitude", "amenities"]
df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)

In [ ]:
# Save the processed dataset
df.to_csv("data/processed/featured_listings.csv", index=False)

In [ ]:
# Verify data
print("Final feature matrix shape:", df.shape)
print("Columns:", df.columns.tolist())

In [58]:
# Convert host_since to datetime with fixed reference
if 'host_since' in df.columns:
    df['host_since'] = pd.to_datetime(df['host_since'], errors='coerce')
    max_host_since = df['host_since'].max()
    df['host_age_days'] = (max_host_since - df['host_since']).dt.days.fillna(0)
    print("Added host_age_days using max host_since as reference.")
else:
    print("'host_since' not found in dataset.")

Added host_age_days using max host_since as reference.


In [59]:
df['host_experience_years'] = df['host_age_days'] / 365.25

## Step 2: Feature Engineering – First Review & Categorical Encoding

**Problem**  
- `last_review` is a date stored as text.
- Categorical features like `room_type` must be encoded.

**Goal**  
Convert `last_review` into `days_since_last_review` and one-hot encode selected categorical variables.

**Approach**  
- Convert `last_review` to datetime.
- One-hot encode `room_type`.

Feature: Days since last review

In [60]:
# Convert last_review to datetime with fixed reference
if 'last_review' in df.columns:
    df['last_review'] = pd.to_datetime(df['last_review'], errors='coerce')
    max_last_review = df['last_review'].max()
    df['days_since_last_review'] = (max_last_review - df['last_review']).dt.days.fillna(0)
    print("Added 'days_since_last_review' using max last_review as reference.")
else:
    print("'last_review' not found in dataset.")

Added 'days_since_last_review' using max last_review as reference.


In [ ]:
df['season'] = pd.to_datetime(df['last_review']).dt.month % 12 // 3  # 0=Winter, 1=Spring, etc.

Feature: Days since last review

In [61]:
# One-hot encode room_type
if 'room_type' in df.columns:
    df['room_type'] = df['room_type'].fillna('Unknown')
    room_dummies = pd.get_dummies(df['room_type'], prefix='room', drop_first=True)
    df = pd.concat([df, room_dummies], axis=1)
    print("One-hot encoded 'room_type'")
else:
    print("'room_type' not found in dataset.")

One-hot encoded 'room_type'


### Step 3: Feature Engineering – Review Scores

**Problem**  
Many review score columns (like `review_scores_rating`, `review_scores_accuracy`, etc.) are numeric but may have missing values.

**Goal**  
- Understand how well-rated each listing is.
- Create an aggregate score or handle missing scores properly.

**Approach**  
- Identify all review score columns.
- Fill missing values with column means or flags.
- (Optional) Create an average score column.


Feature: Review scores & average

In [62]:
# Handle review scores
review_cols = [col for col in df.columns if col.startswith('review_scores_')]
print(f"Found review score columns: {review_cols}")

for col in review_cols:
    if df[col].dtype in ['float64', 'int64']:
        df[col] = df[col].fillna(df[col].median())  # Use median instead of mean
        print(f"Filled missing values in '{col}' with column median.")
    else:
        print(f"Skipping non-numeric column: {col}")

if review_cols:
    df['avg_review_score'] = df[review_cols].mean(axis=1).fillna(0)
    print("Added 'avg_review_score'")

Found review score columns: ['review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value']
Filled missing values in 'review_scores_rating' with column median.
Filled missing values in 'review_scores_accuracy' with column median.
Filled missing values in 'review_scores_cleanliness' with column median.
Filled missing values in 'review_scores_checkin' with column median.
Filled missing values in 'review_scores_communication' with column median.
Filled missing values in 'review_scores_location' with column median.
Filled missing values in 'review_scores_value' with column median.
Added 'avg_review_score'


Feature: Host trust signals

In [63]:
# Convert host flags to binary
for col, new_col in host_flags.items():
    if col in df.columns:
        df[new_col] = df[col].map({'t': 1, 'f': 0}).fillna(0)
        print(f"Converted {col} to binary as {new_col}")
    else:
        print(f"'{col}' not found in dataset.")

Converted host_has_profile_pic to binary as host_has_profile_pic_binary
Converted host_identity_verified to binary as host_identity_verified_binary


Feature: Extract selected amenities into binary columns

In [64]:
# Extract amenities
if 'amenities' in df.columns:
    df['amenities_cleaned'] = df['amenities'].str.replace(r"[{}\"]", "", regex=True)
    df['amenities_list'] = df['amenities_cleaned'].str.lower().str.split(",")
    df['accommodates_bedrooms'] = df['accommodates'] * df['bedrooms']
    df['min_nights_reviews'] = df['minimum_nights'] * df['number_of_reviews']

    def has_amenity(amenity):
        return df['amenities_list'].apply(lambda x: int(amenity in (x or [])) if isinstance(x, list) else 0)

    important_amenities = {
        'Has_Wifi': 'wifi',
        'Has_Kitchen': 'kitchen',
        'Has_Heating': 'heating',
        'Has_TV': 'tv',
        'Has_Essentials': 'essentials',
        'Has_Hair_Dryer': 'hair dryer',
        'Has_Iron': 'iron',
        'Has_Free_Parking': 'parking',
        'Has_Hangers': 'hangers',
        'Has_Laptop_Friendly_Workspace': 'laptop friendly workspace'
    }

    for col, keyword in important_amenities.items():
        df[col] = has_amenity(keyword)
        print(f"Extracted binary column: {col}")
else:
    print("No amenities column found")

Extracted binary column: Has_Wifi
Extracted binary column: Has_Kitchen
Extracted binary column: Has_Heating
Extracted binary column: Has_TV
Extracted binary column: Has_Essentials
Extracted binary column: Has_Hair_Dryer
Extracted binary column: Has_Iron
Extracted binary column: Has_Free_Parking
Extracted binary column: Has_Hangers
Extracted binary column: Has_Laptop_Friendly_Workspace


Drop raw columns

In [65]:
# Drop temporary columns
cols_to_drop = ['amenities_cleaned', 'amenities_list', 'amenities']
df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True)

In [66]:
# Select only engineered features for modeling
selected_features = (numerical_features + categorical_features + list(important_amenities.keys()) + 
                    [col for col in df.columns if col.startswith('room_')] + 
                    ['host_age_days', 'days_since_last_review', 'avg_review_score'] + 
                    list(host_flags.values()))
df = df[selected_features + ['price']]  # Include target for modeling

### Step 5: Save Feature-Engineered Dataset

In [67]:
# Save processed dataset
output_path = Path("data/processed/featured_listings.csv")
try:
    df.to_csv(output_path, index=False)
    print(f"Saved feature-engineered dataset to {output_path} with shape: {df.shape}")
except Exception as e:
    print(f"Error saving to {output_path}: {e}")
    raise

Saved feature-engineered dataset to data\processed\featured_listings.csv with shape: (5090, 32)
